In [1]:
# Explanation: This cell generates all homomorphic images of the 4-uniform tight cycle C_9 with one edge deleted on at most 6 vertices, then excludes those images from the theory.

from collections import defaultdict
from itertools import permutations

forbidden_image_max_size = 6
sdp_target_size = 7
cycle_length = 9
uniformity = 4
source_name = "C9m"


def tight_cycle_edges(n, r=4):
    return [tuple((i + j) % n for j in range(r)) for i in range(n)]


def tight_cycle_minus_edges(n, r=4):
    return tight_cycle_edges(n, r)[:-1]


def canonical_edge_set(num_vertices, edges):
    """Return a canonical representative of an unlabeled 4-graph."""
    best = None
    for perm in permutations(range(num_vertices)):
        relabeled = tuple(sorted({tuple(sorted(perm[v] for v in edge)) for edge in edges}))
        if best is None or relabeled < best:
            best = relabeled
    return best


def generate_homomorphic_images(source_edges, num_source_vertices, max_vertices, r=4):
    """Generate all homomorphic images of the source 4-graph with at most max_vertices vertices, up to isomorphism."""
    images = defaultdict(dict)
    assignment = [None] * num_source_vertices
    assignment[0] = 0

    def complete_edge_is_valid(edge):
        colors = [assignment[v] for v in edge]
        if any(color is None for color in colors):
            return True
        return len(set(colors)) == r

    def partial_assignment_is_valid():
        return all(complete_edge_is_valid(edge) for edge in source_edges)

    def record_image(current_max):
        num_vertices = current_max + 1
        edges = tuple(sorted({tuple(sorted(assignment[v] for v in edge)) for edge in source_edges}))
        key = canonical_edge_set(num_vertices, edges)
        images[num_vertices].setdefault(key, [list(edge) for edge in key])

    def backtrack(pos, current_max):
        if pos == num_source_vertices:
            if partial_assignment_is_valid():
                record_image(current_max)
            return

        upper = min(current_max + 1, max_vertices - 1)
        for color in range(upper + 1):
            assignment[pos] = color
            if partial_assignment_is_valid():
                backtrack(pos + 1, max(current_max, color))
            assignment[pos] = None

    backtrack(1, 0)
    return images


full_cycle_edges = tight_cycle_edges(cycle_length, uniformity)
source_edges = tight_cycle_minus_edges(cycle_length, uniformity)
deleted_edge = full_cycle_edges[-1]

C9m_images_by_size = generate_homomorphic_images(source_edges, cycle_length, forbidden_image_max_size, uniformity)
C9m_homomorphic_images = [
    (num_vertices, edges)
    for num_vertices in sorted(C9m_images_by_size)
    for edges in sorted(C9m_images_by_size[num_vertices].values())
]
C9m_image_theories = [
    FourGraphTheory.p(num_vertices, edges=edges)
    for num_vertices, edges in C9m_homomorphic_images
]

edge = FourGraphTheory(4, edges=[[0, 1, 2, 3]])
FourGraphTheory.exclude(C9m_image_theories)
FGT = FourGraphTheory

print(f"Source graph: {source_name} = C_{cycle_length}^(4)-")
print("deleted edge: ", list(deleted_edge))
print("source edge count: ", len(source_edges))
print(f"Homomorphic images of {source_name} with at most {forbidden_image_max_size} vertices:")
for num_vertices in range(1, forbidden_image_max_size + 1):
    print(f"and size {num_vertices}: ", len(C9m_images_by_size.get(num_vertices, {})))
print("total: ", len(C9m_homomorphic_images))

print("Excluded homomorphic images:")
name_counts = defaultdict(int)
for num_vertices, edges in C9m_homomorphic_images:
    image_index = name_counts[num_vertices]
    name_counts[num_vertices] += 1
    print(f"{source_name}_J{num_vertices}_{image_index} = FourGraphTheory.p({num_vertices}, edges={edges})")

print(f"Number of structures without these {source_name} homomorphic images")
print("and size 5: ", len(FGT.generate(5)))
print("and size 6: ", len(FGT.generate(6)))
# print("and size 7: ", len(FGT.generate(7)))


Source graph: C9m = C_9^(4)-
deleted edge:  [8, 0, 1, 2]
source edge count:  8
Homomorphic images of C9m with at most 6 vertices:
and size 1:  0
and size 2:  0
and size 3:  0
and size 4:  0
and size 5:  2
and size 6:  9
total:  11
Excluded homomorphic images:
C9m_J5_0 = FourGraphTheory.p(5, edges=[[0, 1, 2, 3], [0, 1, 2, 4], [0, 1, 3, 4], [0, 2, 3, 4]])
C9m_J5_1 = FourGraphTheory.p(5, edges=[[0, 1, 2, 3], [0, 1, 2, 4], [0, 1, 3, 4], [0, 2, 3, 4], [1, 2, 3, 4]])
C9m_J6_0 = FourGraphTheory.p(6, edges=[[0, 1, 2, 3], [0, 1, 2, 4], [0, 1, 2, 5], [0, 1, 3, 4], [0, 2, 3, 4]])
C9m_J6_1 = FourGraphTheory.p(6, edges=[[0, 1, 2, 3], [0, 1, 2, 4], [0, 1, 2, 5], [0, 1, 3, 4], [0, 2, 3, 4], [1, 2, 3, 4]])
C9m_J6_2 = FourGraphTheory.p(6, edges=[[0, 1, 2, 3], [0, 1, 2, 4], [0, 1, 2, 5], [0, 1, 3, 4], [0, 2, 3, 4], [1, 2, 3, 5]])
C9m_J6_3 = FourGraphTheory.p(6, edges=[[0, 1, 2, 3], [0, 1, 2, 4], [0, 1, 2, 5], [0, 1, 3, 4], [0, 2, 3, 5]])
C9m_J6_4 = FourGraphTheory.p(6, edges=[[0, 1, 2, 3], [0, 1, 2, 4],

In [2]:
# Explanation: This cell runs the exact flag algebra SDP for the edge density upper bound. No extremal construction is supplied here.

FGT.optimize(
    edge,
    sdp_target_size,
    exact=True,
    file="FourGraph_C9m_7vtx",
    denom=1024*15*3*5,
    slack_threshold=2e-6
)


Base flags generated, their number is 19720
The relevant ftypes are constructed, their number is 6
Block sizes before symmetric/asymmetric change is applied: [2, 19, 558, 342, 220, 85]


Done with mult table for Ftype on 5 points with edges=(0123 0124 0134): : 6it [04:44, 47.34s/it]


Tables finished
Constraints finished
Running sdp without construction. Used block sizes are [2, 9, 10, 19, 539, 33, 309, 36, 184, 21, 64, -19720, -2]
CSDP 6.2.0
Iter:  0 Ap: 0.00e+00 Pobj:  0.0000000e+00 Ad: 0.00e+00 Dobj:  0.0000000e+00 
Iter:  1 Ap: 5.81e-02 Pobj: -7.0273414e+01 Ad: 1.75e-02 Dobj:  8.4485508e-01 
Iter:  2 Ap: 2.72e-01 Pobj: -3.8717444e+02 Ad: 1.04e-01 Dobj:  7.2467656e+00 
Iter:  3 Ap: 3.14e-01 Pobj: -8.3639292e+02 Ad: 4.32e-01 Dobj:  4.0052991e+00 
Iter:  4 Ap: 1.00e+00 Pobj: -2.5082818e+03 Ad: 4.99e-01 Dobj:  1.2549697e+00 
Iter:  5 Ap: 1.00e+00 Pobj: -2.5195415e+03 Ad: 8.61e-01 Dobj:  4.4704402e-02 
Iter:  6 Ap: 1.00e+00 Pobj: -1.8099697e+03 Ad: 8.75e-01 Dobj: -1.8461924e-01 
Iter:  7 Ap: 7.87e-01 Pobj: -1.3520355e+03 Ad: 8.64e-01 Dobj: -2.3376718e-01 
Iter:  8 Ap: 1.67e-01 Pobj: -1.2726467e+03 Ad: 7.13e-01 Dobj: -2.3641053e-01 
Iter:  9 Ap: 7.63e-01 Pobj: -1.1384096e+03 Ad: 5.70e-01 Dobj: -2.4750804e-01 
Iter: 10 Ap: 6.18e-01 Pobj: -1.3192894e+03 Ad: 5.49e-01 Dob

 17%|███████▎                                    | 1/6 [07:22<36:51, 442.22s/it]

Rounded X matrix ((5, Ftype on 3 points with edges=(), 7), 0) is not semidefinite: -9.116708299107261e-05


...

100%|███████████████████████████████████████████| 11/11 [00:12<00:00,  1.12s/it]


Calculating resulting bound


100%|█████████████████████████████████████████| 6/6 [3:19:19<00:00, 1993.18s/it]


The rounded result is -693253/1612800


693253/1612800